In [ ]:
import asyncio
import time
from typing import List

import uvicorn
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field

app = FastAPI(
    title="DZ-3",
    description="Асинхронноcть в Python",
    version="1.0.0",
)

class CalcResult(BaseModel):
    number: float
    square: float
    delay: float
    time: float

class CalcReq(BaseModel):
    numbers: List[float] = Field(
        ...,
        description="Список чисел, для которых считаем квадраты",
        examples=[[5, 3, 10]],
    )
    delays: List[float] = Field(
        ...,
        description="Список задержек в секундах",
        examples=[[1, 2, 0.5]],
    )

class CalcRes(BaseModel):
    results: List[CalcResult]
    total_time: float
    parallel_faster_than_sequential: bool


async def calc_square(number: float, delay: float) -> CalcResult:

    start = time.perf_counter()
    await asyncio.sleep(delay)
    elapsed = time.perf_counter() - start

    return CalcResult(
        number=number,
        square=number ** 2,
        delay=delay,
        time=round(elapsed, 2),
    )

# API

@app.post("/calculate/", response_model=CalcRes)
async def calculate(request: CalcReq) -> CalcRes:
    if len(request.numbers) != len(request.delays):
        raise HTTPException(
            status_code=400,
            detail="Списки numbers и delays должны быть одинаковой длины",
        )
    if any(d < 0 for d in request.delays):
        raise HTTPException(
            status_code=400,
            detail="Задержки не могут быть отрицательными",
        )

    start = time.perf_counter()

    results = await asyncio.gather(
        *(
            calc_square(number, delay)
            for number, delay in zip(request.numbers, request.delays)
        )
    )

    total_time = time.perf_counter() - start

    return CalcRes(
        results=list(results),
        total_time=round(total_time, 2),
        parallel_faster_than_sequential=total_time < sum(request.delays),
    )


if __name__ == "__main__":
    uvicorn.run("main:app", host="127.0.0.1", port=8003)

INFO:     Will watch for changes in these directories: ['c:\\Users\\graft\\MAGA\\python_braps\\homew\\python_braps']
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)
INFO:     Started reloader process [32500] using StatReload
INFO:     Stopping reloader process [32500]
